# Vector算子开发实践

通过前面的学习，我们已经掌握了使用AscendC开发算子的完整方法，尤其是Tiling相关知识。本实践在核函数实践的基础上，需要实现一个泛化的 Sigmoid 算子，并且完成精度验证。

Sigmoid 计算公式：

                           sigmoid(x) = 1/(1 + exp(-x))

实践要求：
- 完成`Sigmoid`的Kernel侧计算逻辑。
- 在Host侧直接完成泛化Tiling计算。
- 在Host侧使用 `sigmoid_custom<<<...>>>` 直接调用核函数。
- 支持 float/half 类型输入输出。
- 支持以下调用case并运行成功：
  - case1：输入 x 的 shape 为 `(7, 83)`，类型为 float32。
  - case2：输入 x 的 shape 为 `(1024, 1024)`，类型为 float16。
  - case3：输入 x 的 shape 为 `(999, 999)`，类型为 float32。

请开始你的实践，体验完整开发过程。

---
## 环境准备

In [ ]:
!mkdir -p Sources/02.08

import os
import subprocess
from pathlib import Path

set_env = os.environ.get("ASCEND_TOOLKIT_HOME", "/usr/local/Ascend/cann") + "/set_env.sh"
if not Path(set_env).exists():
    set_env = "/usr/local/Ascend/cann/set_env.sh"

result = subprocess.run(
    ["bash", "-lc", f"source {set_env} && env"],
    capture_output=True,
    text=True,
    check=True,
)
for line in result.stdout.strip().split("\n"):
    if "=" in line and not line.startswith(("#", " ")):
        key, value = line.split("=", 1)
        os.environ[key] = value

print("Environment initialization process completed successfully.")

---
## 目录结构介绍

本次实践代码按照如下目录结构存放：

```
├── 02.08
│   ├── scripts
│   │   ├── gen_data.py                // 输入数据和真值数据生成脚本
│   │   └── verify_result.py           // 验证输出数据和真值数据是否一致的验证脚本
│   ├── CMakeLists.txt                 // 编译工程文件
│   ├── data_utils.h                   // 数据读入写出函数
│   ├── sigmoid_custom.asc             // host侧调用 + kernel侧实现
│   └── run.sh                         // 运行脚本
```

In [ ]:
!cp -rf ./src/02.08 ./Sources

---
## Sigmoid 直调实现
该实现将 Host 侧 Tiling、Kernel 侧 Sigmoid 计算、`<<<>>>` 调用以及三组 case 验证放在同一个 `sigmoid_custom.asc` 文件中。
请填充下列TODO内容。

In [ ]:
%%writefile Sources/02.08/sigmoid_custom.asc

#include <algorithm>
#include <cmath>
#include <cstdint>
#include <iostream>
#include <type_traits>
#include <vector>

#include "acl/acl.h"
#include "kernel_operator.h"
#include "tiling/platform/platform_ascendc.h"
#include "data_utils.h"

constexpr int32_t BUFFER_NUM = 2;

struct SigmoidCustomTilingData {
    uint32_t smallCoreDataNum;
    uint32_t bigCoreDataNum;
    uint32_t finalBigTileNum;
    uint32_t finalSmallTileNum;
    uint32_t tileDataNum;
    uint32_t smallTailDataNum;
    uint32_t bigTailDataNum;
    uint32_t tailBlockNum;
};

int64_t GetShapeSize(const std::vector<int64_t> &shape)
{
    int64_t shapeSize = 1;
    for (auto dim : shape) {
        shapeSize *= dim;
    }
    return shapeSize;
}

template <typename T>
void CalcTiling(uint32_t inputNum, SigmoidCustomTilingData &tiling, uint32_t &numBlocks)
{
    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();

    constexpr uint32_t BLOCK_SIZE = 32;
    const uint32_t typeLength = static_cast<uint32_t>(sizeof(T));
    const uint32_t inputLength = inputNum * typeLength;
    const uint32_t inputLengthAlign32 = ((inputLength + BLOCK_SIZE - 1) / BLOCK_SIZE) * BLOCK_SIZE;
    const uint32_t inputBlockNum = std::max(inputLengthAlign32 / BLOCK_SIZE, 1U);

    uint32_t coreNum = ascendcPlatform->GetCoreNum();
    numBlocks = coreNum;

    const uint32_t everyCoreInputBlockNumBase = inputBlockNum / coreNum;
    const uint32_t tailBlockNum = inputBlockNum % coreNum;

    uint64_t ubSize = 0;
    ascendcPlatform->GetCoreMemSize(platform_ascendc::CoreMemType::UB, ubSize);
    const uint32_t ubDataNumber = 2;
    const uint32_t tileBlockNum = static_cast<uint32_t>((ubSize / BLOCK_SIZE / BUFFER_NUM) / ubDataNumber);
    const uint32_t tileDataNum = tileBlockNum * BLOCK_SIZE / typeLength;

    const uint32_t smallCoreDataNum = everyCoreInputBlockNumBase * BLOCK_SIZE / typeLength;
    const uint32_t smallTileNum = everyCoreInputBlockNumBase / tileBlockNum;
    const uint32_t finalSmallTileNum = (everyCoreInputBlockNumBase % tileBlockNum) == 0 ? smallTileNum : smallTileNum + 1;
    uint32_t smallTailDataNum = smallCoreDataNum - tileDataNum * smallTileNum;
    smallTailDataNum = smallTailDataNum == 0 ? tileDataNum : smallTailDataNum;

    const uint32_t bigCoreInputBlockNum = everyCoreInputBlockNumBase + 1;
    const uint32_t bigCoreDataNum = bigCoreInputBlockNum * BLOCK_SIZE / typeLength;
    const uint32_t bigTileNum = bigCoreInputBlockNum / tileBlockNum;
    const uint32_t finalBigTileNum = (bigCoreInputBlockNum % tileBlockNum) == 0 ? bigTileNum : bigTileNum + 1;
    uint32_t bigTailDataNum = bigCoreDataNum - tileDataNum * bigTileNum;
    bigTailDataNum = bigTailDataNum == 0 ? tileDataNum : bigTailDataNum;

    tiling.smallCoreDataNum = smallCoreDataNum;
    tiling.bigCoreDataNum = bigCoreDataNum;
    tiling.finalBigTileNum = finalBigTileNum;
    tiling.finalSmallTileNum = finalSmallTileNum;
    tiling.tileDataNum = tileDataNum;
    tiling.smallTailDataNum = smallTailDataNum;
    tiling.bigTailDataNum = bigTailDataNum;
    tiling.tailBlockNum = tailBlockNum;
}

template<typename T>
class KernelSigmoid {
public:
    __aicore__ inline KernelSigmoid() {}

    __aicore__ inline void Init(GM_ADDR x, GM_ADDR y, uint32_t smallCoreDataNum, uint32_t bigCoreDataNum,
        uint32_t finalBigTileNum, uint32_t finalSmallTileNum, uint32_t tileDataNum, uint32_t smallTailDataNum,
        uint32_t bigTailDataNum, uint32_t tailBlockNum)
    {
        uint32_t coreNum = AscendC::GetBlockIdx();
        uint32_t globalBufferIndex = bigCoreDataNum * AscendC::GetBlockIdx();
        this->tileDataNum = tileDataNum;

        if (coreNum < tailBlockNum) {
            this->coreDataNum = bigCoreDataNum;
            this->tileNum = finalBigTileNum;
            this->tailDataNum = bigTailDataNum;
        } else {
            this->coreDataNum = smallCoreDataNum;
            this->tileNum = finalSmallTileNum;
            this->tailDataNum = smallTailDataNum;
            globalBufferIndex -= (bigCoreDataNum - smallCoreDataNum) * (AscendC::GetBlockIdx() - tailBlockNum);
        }

        xGm.SetGlobalBuffer((__gm__ T*)x + globalBufferIndex, this->coreDataNum);
        yGm.SetGlobalBuffer((__gm__ T*)y + globalBufferIndex, this->coreDataNum);
        pipe.InitBuffer(inQueueX, BUFFER_NUM, this->tileDataNum * sizeof(T));
        pipe.InitBuffer(outQueueY, BUFFER_NUM, this->tileDataNum * sizeof(T));
    }

    __aicore__ inline void Process()
    {
        for (int32_t i = 0; i < static_cast<int32_t>(this->tileNum); ++i) {
            this->processDataNum = (i == static_cast<int32_t>(this->tileNum) - 1) ? this->tailDataNum : this->tileDataNum;
            CopyIn(i);
            Compute(i);
            CopyOut(i);
        }
    }

private:
    __aicore__ inline void CopyIn(int32_t progress)
    {
        AscendC::LocalTensor<T> xLocal = inQueueX.AllocTensor<T>();
        AscendC::DataCopy(xLocal, xGm[progress * this->tileDataNum], this->processDataNum);
        inQueueX.EnQue(xLocal);
    }

    __aicore__ inline void Compute(int32_t progress)
    {
        AscendC::LocalTensor<T> xLocal = inQueueX.DeQue<T>();
        AscendC::LocalTensor<T> yLocal = outQueueY.AllocTensor<T>();
        AscendC::Sigmoid(yLocal, xLocal, this->processDataNum);
        outQueueY.EnQue<T>(yLocal);
        inQueueX.FreeTensor(xLocal);
    }

    __aicore__ inline void CopyOut(int32_t progress)
    {
        AscendC::LocalTensor<T> yLocal = outQueueY.DeQue<T>();
        AscendC::DataCopy(yGm[progress * this->tileDataNum], yLocal, this->processDataNum);
        outQueueY.FreeTensor(yLocal);
    }

private:
    AscendC::TPipe pipe;
    AscendC::TQue<AscendC::QuePosition::VECIN, BUFFER_NUM> inQueueX;
    AscendC::TQue<AscendC::QuePosition::VECOUT, BUFFER_NUM> outQueueY;
    AscendC::GlobalTensor<T> xGm;
    AscendC::GlobalTensor<T> yGm;
    uint32_t coreDataNum = 0;
    uint32_t tileNum = 0;
    uint32_t tileDataNum = 0;
    uint32_t tailDataNum = 0;
    uint32_t processDataNum = 0;
};

template <typename T>
 __global__ __vector__ void sigmoid_custom(GM_ADDR x, GM_ADDR y, SigmoidCustomTilingData tiling)
{
    KernelSigmoid<T> op;
    op.Init(x, y, tiling.smallCoreDataNum, tiling.bigCoreDataNum, tiling.finalBigTileNum, tiling.finalSmallTileNum,
        tiling.tileDataNum, tiling.smallTailDataNum, tiling.bigTailDataNum, tiling.tailBlockNum);
    op.Process();
}

template <typename T>
void RunCase(const char *caseName, const std::vector<int64_t> &shape, int caseIndex)
{
    printf("Testcase: %s start ======\n", caseName);

    const uint32_t inputNum = static_cast<uint32_t>(GetShapeSize(shape));
    size_t dataSize = inputNum * sizeof(T);
    std::string srcFile = "./input/src_gm_case" + std::to_string(caseIndex) + ".bin";
    std::string dstFile = "./output/output_case" + std::to_string(caseIndex) + ".bin";

    int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    aclInit(nullptr);
    aclrtSetDevice(deviceId);
    aclrtCreateStream(&stream);

    auto ascendcPlatform = platform_ascendc::PlatformAscendCManager::GetInstance();
    SigmoidCustomTilingData tiling{};
    uint32_t numBlocks = 0;
    CalcTiling<T>(inputNum, tiling, numBlocks);

    uint8_t* xHost;
    uint8_t *xDevice;
    aclrtMallocHost((void**)(&xHost), dataSize);
    aclrtMalloc((void**)&xDevice, dataSize, ACL_MEM_MALLOC_HUGE_FIRST);
    ReadFile(srcFile, dataSize, xHost, dataSize);
    aclrtMemcpy(xDevice, dataSize, xHost, dataSize, ACL_MEMCPY_HOST_TO_DEVICE);

    uint8_t *yHost;
    uint8_t *yDevice;
    aclrtMallocHost(reinterpret_cast<void **>(&yHost), dataSize);
    aclrtMalloc(reinterpret_cast<void **>(&yDevice), dataSize, ACL_MEM_MALLOC_HUGE_FIRST);

    sigmoid_custom<T><<<numBlocks, nullptr, stream>>>(xDevice, yDevice, tiling);
    aclrtSynchronizeStream(stream);

    aclrtMemcpy(yHost, dataSize, yDevice, dataSize, ACL_MEMCPY_DEVICE_TO_HOST);
    WriteFile(dstFile, yHost, dataSize);

    aclrtFree(xDevice);
    aclrtFree(yDevice);
    aclrtFreeHost(xHost);
    aclrtFreeHost(yHost);
    aclrtDestroyStream(stream);
    aclrtResetDevice(deviceId);
    aclFinalize();
}

int32_t main(int32_t argc, char *argv[])
{
    RunCase<float>("case0 float32 shape=(7,83)", {7, 83}, 0);
    RunCase<half>("case1 float16 shape=(1024,1024)", {1024, 1024}, 1);
    RunCase<float>("case2 float32 shape=(999,999)", {999, 999}, 2);
    return 0;
}

---
## 编译运行
运行后应看到三组 case 均输出 `test pass`。

In [ ]:
!cd Sources/02.08 && bash run.sh

---
## 查看答案
实现方式不唯一，这里给出的答案仅供参考。

In [ ]:
!cat ./answer/02.08/sigmoid_custom.asc